<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/ASA/EX_harmonic_mistuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harmonicity as a grouping cue: the mistuned harmonic

A periodic sound such as a voiced vowel or a musical note consists of **harmonics**: components at integer multiples of a fundamental frequency $f_0$. The auditory system uses this regularity for *simultaneous* grouping: components that fit one harmonic series are fused into a single sound with one pitch and one timbre.

If one component is moved off its harmonic position (**mistuned**), it stops fitting the series. With enough mistuning it can be heard out as a separate tone, while the rest of the complex still forms one sound. Another cue that competes with harmonicity is **common onset**: a component that starts earlier or later than the rest also tends to be heard separately.

**Use headphones** and set the volume to a comfortable level before playing anything.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Audio, display

FS = 44100  # sampling rate (Hz)

def ramp(n_samples, n_ramp):
    """Raised-cosine on/off ramps to avoid clicks."""
    w = np.ones(n_samples)
    r = 0.5 * (1 - np.cos(np.linspace(0, np.pi, n_ramp)))
    w[:n_ramp] = r
    w[-n_ramp:] = r[::-1]
    return w

def component_freqs(f0, n_harm, k, mistune_pct):
    """Harmonic frequencies; harmonic k is shifted by mistune_pct percent."""
    f = f0 * np.arange(1, n_harm + 1, dtype=float)
    f[k - 1] *= 1 + mistune_pct / 100
    return f

def _raw(f0, n_harm, k, mistune_pct, dur, onset_delay_ms, only_k):
    n = int(dur * FS)
    t = np.arange(n) / FS
    freqs = component_freqs(f0, n_harm, k, mistune_pct)
    x = np.zeros(n)
    for i, f in enumerate(freqs, start=1):
        if only_k and i != k:
            continue
        s = np.sin(2 * np.pi * f * t)
        if i == k and onset_delay_ms > 0:
            s[:int(onset_delay_ms / 1000 * FS)] = 0
        x += s
    return x * ramp(n, int(0.02 * FS))      # 20 ms ramps

def make_complex(f0=200, n_harm=12, k=4, mistune_pct=0.0, dur=1.0, onset_delay_ms=0.0, only_k=False):
    """Equal-amplitude harmonic complex. Harmonic k can be mistuned and/or start late.
    If only_k is True, only the (possibly mistuned) component k is returned.
    All versions share one scale factor (set by the in-tune complex), so levels are comparable
    and nothing clips."""
    ref_peak = np.max(np.abs(_raw(f0, n_harm, k, 0.0, dur, 0.0, False)))
    scale = 0.5 / max(ref_peak, 1e-9)
    return scale * _raw(f0, n_harm, k, mistune_pct, dur, onset_delay_ms, only_k)

def spectrum_plot(f0, n_harm, k, mistune_pct):
    freqs = component_freqs(f0, n_harm, k, mistune_pct)
    fig, ax = plt.subplots(figsize=(9, 2.6))
    for i, f in enumerate(freqs, start=1):
        ax.vlines(f, 0, 1, color='tab:red' if i == k else 'tab:blue', lw=3)
    ax.vlines(f0 * k, 0, 1, color='tab:red', lw=1, ls=':')   # where harmonic k would be
    ax.set_xlabel('frequency (Hz)'); ax.set_yticks([])
    ax.set_title(f'harmonic {k} (red) mistuned by {mistune_pct:+.1f} %: {freqs[k-1]:.1f} Hz instead of {f0*k:.1f} Hz')
    plt.tight_layout(); plt.show()


## 1. Explore one stimulus

Press **Run Interact** after changing the sliders. Three sounds are shown: the complex with the chosen component mistuned, the same complex in tune (reference), and the mistuned component alone.

Things to try:
- **mistuning** $0 \to 1 \to 2 \to 4 \to 8$ %: when does the component start to stand out?
- **harmonic number** $k$: is it easier to hear out low harmonics (e.g. 2–5) or high ones (e.g. 10)?
- **onset delay**: with 0 % mistuning (a perfect harmonic), delay the component by 50–200 ms. Does it separate from the rest?

In [2]:
def explore(f0=200, n_harm=12, k=4, mistune_pct=8.0, onset_delay_ms=0, dur=1.0):
    k = min(k, n_harm)
    spectrum_plot(f0, n_harm, k, mistune_pct)
    print('1) complex with mistuned / delayed component')
    display(Audio(make_complex(f0, n_harm, k, mistune_pct, dur, onset_delay_ms), rate=FS))
    print('2) reference: all harmonics in tune, no delay')
    display(Audio(make_complex(f0, n_harm, k, 0.0, dur, 0.0), rate=FS))
    print('3) the (mistuned) component alone')
    display(Audio(make_complex(f0, n_harm, k, mistune_pct, dur, 0.0, only_k=True), rate=FS))

widgets.interact_manual(
    explore,
    f0=widgets.IntSlider(value=200, min=100, max=400, step=10, description='f0 (Hz)'),
    n_harm=widgets.IntSlider(value=12, min=6, max=20, description='harmonics'),
    k=widgets.IntSlider(value=4, min=1, max=20, description='mistuned k'),
    mistune_pct=widgets.FloatSlider(value=8.0, min=-16, max=16, step=0.5, description='mistuning %'),
    onset_delay_ms=widgets.IntSlider(value=0, min=0, max=300, step=10, description='delay (ms)'),
    dur=widgets.FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description='duration (s)'),
);

interactive(children=(IntSlider(value=200, description='f0 (Hz)', max=400, min=100, step=10), IntSlider(value=…

## 2. Lecture demo: increasing mistuning

One audio file plays the complex with harmonic $k$ mistuned by 0, 1, 2, 4, 8 and 16 %, with short pauses between the sounds. Ask students to raise a hand when they first hear a separate tone.

In [3]:
def demo_series(f0=200, n_harm=12, k=4, steps=(0, 1, 2, 4, 8, 16), dur=0.8, gap=0.6):
    pause = np.zeros(int(gap * FS))
    parts = []
    for m in steps:
        parts += [make_complex(f0, n_harm, k, m, dur), pause]
    print('Order of the sounds (mistuning of harmonic %d, in %%): %s' % (k, list(steps)))
    display(Audio(np.concatenate(parts), rate=FS))

demo_series(k=4)

Order of the sounds (mistuning of harmonic 4, in %): [0, 1, 2, 4, 8, 16]


In [ ]:
# Same demo for a high harmonic: compare with the one above
demo_series(k=10)

## 3. Lecture demo: common onset competes with harmonicity

All harmonics are **in tune** (0 % mistuning), but harmonic $k$ starts later by 0, 50, 100, 200 ms.

In [4]:
def demo_onset(f0=200, n_harm=12, k=4, delays=(0, 50, 100, 200), dur=1.0, gap=0.6):
    pause = np.zeros(int(gap * FS))
    parts = []
    for d in delays:
        parts += [make_complex(f0, n_harm, k, 0.0, dur, d), pause]
    print('Onset delay of harmonic %d (ms): %s' % (k, list(delays)))
    display(Audio(np.concatenate(parts), rate=FS))

demo_onset()

Onset delay of harmonic 4 (ms): [0, 50, 100, 200]


## Discussion questions

1. Why does a mistuned component stand out only when it is mistuned by more than a few percent?
2. Low harmonics are **resolved** by cochlear filtering (each excites its own place), high harmonics are not. How could this explain differences between $k=4$ and $k=10$?
3. Which cue wins when harmonicity and common onset disagree? Which result in section 3 supports your answer?
4. Where would this fit in the *primitive vs. schema-based* grouping scheme of Bregman?

## Notes and cautions

- The perceptual effect depends on $f_0$, the harmonic number, level, duration and listener. Treat the percentages above as a demonstration, not as thresholds. Classic experiments on mistuned harmonics include Moore, Peters & Glasberg (1985, *J. Acoust. Soc. Am.*); please check the original for the numbers before quoting them to students.
- Slight mistuning can also shift the pitch of the whole complex or add roughness before the component pops out as a separate tone.
- Reproduction through laptop speakers can distort low-level components, so headphones are recommended.